# Introduction

Customer Behaviour and it’s prediction lies 
at the core of every Business Model. From 
Stock Exchange, e-Commerce and 
Automobile to even Presidential Elections, 
predictions serve a great purpose. Most of 
these predictions are based on the data 
available about a person’s activity either 
online or in-person.

## Problem Statement

For the purpose of this Project we will be creating one such Recommendation Engine 
from the ground-up, where every single user, based on there area of interest and ratings, 
would be recommended a list of movies that are best suited for them.


## Dataset Information

1. **ID** –Contains the separate keys for  customer and movies. 
2. **Rating** – A section contains the user ratings for all the movies.
3. **Genre** –Highlights the category of the 
4. **Movie Name** –Name of the movie with respect to the movie id.

## Steps We will take

In this notebook, we will follow a structured approach to wrangle the movie datasets. The steps include:

1. **Data Gathering**: Import necessary libraries (pandas, numpy) and load the datasets.
2. **Data Assessing & Cleaning**: Manually export for review and programmatically check structure, missing values, duplicates, and statistics.
3. **Validation and Merging**: Re-check the cleaned data.
4. **Exploratory Data Analysis**: Finding the hidden patter inside data distribution.
5. **Model fitting**: fit the data on the best suited model.
6. **Model Evaluation & validation**: check the model accuracy and performance on varius metric
7. **Model Deployement**: Simple flask app to check the model performance

### Data Gathering:

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [61]:
from pathlib import Path

# Split only on the first two commas; remaining commas belong to the movie title.
rows = []
with Path('movie_titles.csv').open(encoding='ISO-8859-1', newline='') as file:
    for line in file:
        fields = line.rstrip('\r\n').split(',', 2)
        if len(fields) == 3:
            rows.append(fields)

df_movies = pd.DataFrame(rows, columns=['movie_id', 'year', 'movie_name'])
df_movies['movie_id'] = pd.to_numeric(df_movies['movie_id'], errors='coerce')
df_movies['year'] = pd.to_numeric(df_movies['year'], errors='coerce')

df_movies.tail()

,movie_id,year,movie_name
17765,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17766,17767,2004.0,Fidel Castro: American Experience
17767,17768,2000.0,Epoch
17768,17769,2003.0,The Company
17769,17770,2003.0,Alien Hunter


**Note:**  
- There is a Data importing issue in raw data, since there are movies with comma. Hence `movie_name` col is fixed.

In [62]:
netflix_df = pd.read_csv('combinedNetflixData.txt', header=None, sep = '\t')

In [63]:
netflix_df.head()

,0
0,1:
1,"1488844,3,2005-09-06"
2,"822109,5,2005-05-13"
3,"885013,4,2005-10-19"
4,"30878,4,2005-12-26"


In [64]:
netflix_df.rename(columns={0: 'customer_id'}, inplace=True)

In [65]:
mask = netflix_df['customer_id'].astype(str).str.match(r"^\d+:$")

In [66]:
# Extract movie ID from those rows
netflix_df["movie_id"] = netflix_df["customer_id"].where(mask).str.replace(":", "", regex=False)

In [67]:
netflix_df.head()

,customer_id,movie_id
0,1:,1
1,"1488844,3,2005-09-06",NaN
2,"822109,5,2005-05-13",NaN
3,"885013,4,2005-10-19",NaN
4,"30878,4,2005-12-26",NaN


In [68]:
# Propagate movie ID to subsequent rows
netflix_df["movie_id"] = netflix_df["movie_id"].ffill()

# Remove the movie-header rows
netflix_df = netflix_df[~mask].copy()

In [69]:
# Convert columns to numeric types
netflix_df["movie_id"] = netflix_df["movie_id"].astype(int)
print(netflix_df)

                   customer_id  movie_id
1         1488844,3,2005-09-06         1
2          822109,5,2005-05-13         1
3          885013,4,2005-10-19         1
4           30878,4,2005-12-26         1
5          823519,3,2004-05-03         1
...                        ...       ...
24058258  2591364,2,2005-02-16      4499
24058259  1791000,2,2005-02-10      4499
24058260   512536,5,2005-07-27      4499
24058261   988963,3,2005-12-20      4499
24058262  1704416,3,2004-06-02      4499

[24053764 rows x 2 columns]


In [70]:
netflix_df[["customer_id", "rating", "date"]] = (
    netflix_df["customer_id"].str.split(",", expand=True)
)

netflix_df["customer_id"] = netflix_df["customer_id"].astype(int)
netflix_df["rating"] = netflix_df["rating"].astype(int)
netflix_df["date"] = pd.to_datetime(netflix_df["date"])

In [71]:
netflix_df['rating'] = pd.to_numeric(netflix_df['rating'], errors='coerce')

In [72]:
netflix_df.drop(columns=['date'], inplace=True)

**Note**
- 'CombinedNetflixData.txt' has messy data problem. 'movie_id' has been seperated.
- 'Date' column has been removed from since we are dealing with tabular data.

### Data Assessing & Cleaning

- Over Here we are going to perform two type of Data assessing.
1. Mannual Assessing (using Excel)
2. Programatic Assessing (using Pandas or numpy)

- Afterwards we will write all the issues, in order to clean our data.


In [73]:
netflix_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24053764 entries, 1 to 24058262
Data columns (total 3 columns):
 #   Column       Dtype
---  ------       -----
 0   customer_id  int64
 1   movie_id     int64
 2   rating       int64
dtypes: int64(3)
memory usage: 734.1 MB


In [95]:
netflix_df.shape

(24053764, 3)

In [99]:
netflix_df['movie_id'].unique().shape

(4499,)

In [100]:
df_movies['movie_id'].unique().shape

(17770,)

#### Issues with the dataset found

**Data - `movie_titles.csv`**

- **validity issue**: Some movies are having with comma which creating issue. Excel and pandas considering it as new column. Hence importing only 1st column.
- **validity issue**: Some movies name have speacial charecters, need to remove before apply ml.
- **completion issues**: multiple movies has `year` missing. since null is less than 10, need manual filling.
- **validity issue**: Column are in diffrent format.



**Data - `combinedNetflixData.txt`**

- **Messy Data**: Movie id are given in the first row followed by customer_id, rating and date.
- **validity issue**: columns are found in string format.
- **completion issue**: There is only 4499 movies rating available out of total 17770 movies.




In [74]:
#**validity issue**: Some movies name have speacial charecters, need to remove before 
df_movies['movie_name'] = df_movies['movie_name'].str.replace(r'[\W_]', '', regex=True)

In [75]:
mask = df_movies['year'].isnull()

In [76]:
df_movies[mask]

,movie_id,year,movie_name
4387,4388,NaN,AncientCivilizationsRomeandPompeii
4793,4794,NaN,AncientCivilizationsLandofthePharaohs
7240,7241,NaN,AncientCivilizationsAthensandGreece
10781,10782,NaN,RotiKapadaAurMakaan
15917,15918,NaN,HoteHotePyaarHoGaya
16677,16678,NaN,JimmyHollywood
17666,17667,NaN,ErosDanceDhamaka


In [77]:
indexing = df_movies[mask].index

In [78]:
df_movies.loc[indexing, 'year'] = ['2001', '1995', '2001', '1974', '1999', '1994', '2022' ]

C:\Users\shubh\AppData\Local\Temp\ipykernel_30516\91213435.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2001', '1995', '2001', '1974', '1999', '1994', '2022']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_movies.loc[indexing, 'year'] = ['2001', '1995', '2001', '1974', '1999', '1994', '2022' ]


In [79]:
df_movies.loc[indexing, 'year']

4387     2001
4793     1995
7240     2001
10781    1974
15917    1999
16677    1994
17666    2022
Name: year, dtype: object

In [86]:
df_movies['year'] = df_movies['year'].astype(int)

In [88]:
df_movies['year'].value_counts()

year
2004    1436
2002    1310
2003    1271
2000    1234
2001    1186
        ... 
1923       2
1914       2
1909       1
1896       1
2022       1
Name: count, Length: 95, dtype: int64

In [89]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17770 entries, 0 to 17769
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   movie_id    17770 non-null  int64 
 1   year        17770 non-null  int64 
 2   movie_name  17770 non-null  object
dtypes: int64(2), object(1)
memory usage: 416.6+ KB


### Validation and Merging

In [ ]:
#Since the both dataframe are ready, we can merge them to get the final dataset for analysis. 
# We will use inner join to get the common records from both the dataframes.

df = pd.merge(df_movies, netflix_df, how='inner', on='movie_id')


In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24053764 entries, 0 to 24053763
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   movie_id     int64 
 1   year         int64 
 2   movie_name   object
 3   customer_id  int64 
 4   rating       int64 
dtypes: int64(4), object(1)
memory usage: 917.6+ MB


### Export Clean Data

In [105]:
df.to_csv('final_dataset.csv', index=False)

### Exploratory Data Analysis

### Model fitting

### Model Evaluation & validation

### Model Deployement